In [61]:
# Load packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import session_info

In [62]:
# Load data
frogID_data = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-02/frogID_data.csv')
frog_names = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-02/frog_names.csv')


In [63]:
df = frogID_data.merge(frog_names, on='scientificName', how='left')
df = df.iloc[:, [2, 3, 5, 6, 7, 10, 11]]
df.head()

,decimalLatitude,decimalLongitude,eventDate,eventTime,timezone,stateProvince,subfamily
0,-28.5,153.1,2023-01-01,11:18:32,GMT+1100,New South Wales,Myobatrachid
1,-33.7,151.2,2023-01-02,20:39:30,GMT+1100,New South Wales,NaN
2,-28.7,152.7,2023-01-02,21:30:07,GMT+1100,New South Wales,Myobatrachid
3,-28.7,152.7,2023-01-02,21:30:07,GMT+1100,New South Wales,Myobatrachid
4,-28.7,152.7,2023-01-02,21:30:07,GMT+1100,New South Wales,Hylid


In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136625 entries, 0 to 136624
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   decimalLatitude   136625 non-null  float64
 1   decimalLongitude  136625 non-null  float64
 2   eventDate         136625 non-null  object 
 3   eventTime         136625 non-null  object 
 4   timezone          136625 non-null  object 
 5   stateProvince     136625 non-null  object 
 6   subfamily         127467 non-null  object 
dtypes: float64(2), object(5)
memory usage: 7.3+ MB


In [65]:
print("Unique timezone values:")
print(df['timezone'].value_counts())

Unique timezone values:
timezone
UTC         57503
GMT+1100    36702
GMT+1000    25284
GMT+0800    10660
GMT+0930     5476
GMT+1030     1000
Name: count, dtype: int64


In [66]:
# Convert timezones to consistent format
tz_norm = (
    df['timezone']
    .str.upper().str.strip()
    .str.replace(r"^GMT([+-]\d{4})$", r"\1", regex=True)  # GMT+1100 -> +1100
    .str.replace(r"^UTC$", "+0000", regex=True)           # UTC -> +0000
)

# Concatenate the date, time, and timezone columns
df['datetime_str'] = (
    df['eventDate'].astype(str).str.strip() + " " +
    df['eventTime'].astype(str).str.strip() + " " +
    tz_norm
)

# Convert the datetime_str column to a datetime
df['timestamp'] = pd.to_datetime(
    df['datetime_str'], 
    format='%Y-%m-%d %H:%M:%S %z',
    errors='coerce'
)

# Convert the datetimes to UTC
df['timestamp_utc'] = df['timestamp'].apply(lambda x: x.tz_convert('UTC') if pd.notna(x) else pd.NaT)

df.head()


/var/folders/7d/cvrlcn892fd5f01pqw2vmql40000gn/T/ipykernel_29904/2053044271.py:17: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['timestamp'] = pd.to_datetime(


,decimalLatitude,decimalLongitude,eventDate,eventTime,timezone,stateProvince,subfamily,datetime_str,timestamp,timestamp_utc
0,-28.5,153.1,2023-01-01,11:18:32,GMT+1100,New South Wales,Myobatrachid,2023-01-01 11:18:32 +1100,2023-01-01 11:18:32+11:00,2023-01-01 00:18:32+00:00
1,-33.7,151.2,2023-01-02,20:39:30,GMT+1100,New South Wales,NaN,2023-01-02 20:39:30 +1100,2023-01-02 20:39:30+11:00,2023-01-02 09:39:30+00:00
2,-28.7,152.7,2023-01-02,21:30:07,GMT+1100,New South Wales,Myobatrachid,2023-01-02 21:30:07 +1100,2023-01-02 21:30:07+11:00,2023-01-02 10:30:07+00:00
3,-28.7,152.7,2023-01-02,21:30:07,GMT+1100,New South Wales,Myobatrachid,2023-01-02 21:30:07 +1100,2023-01-02 21:30:07+11:00,2023-01-02 10:30:07+00:00
4,-28.7,152.7,2023-01-02,21:30:07,GMT+1100,New South Wales,Hylid,2023-01-02 21:30:07 +1100,2023-01-02 21:30:07+11:00,2023-01-02 10:30:07+00:00


In [67]:
# Remove unneccesary columns
df = df.iloc[:, [0, 1, 5, 6, 9]]
df.head()

,decimalLatitude,decimalLongitude,stateProvince,subfamily,timestamp_utc
0,-28.5,153.1,New South Wales,Myobatrachid,2023-01-01 00:18:32+00:00
1,-33.7,151.2,New South Wales,NaN,2023-01-02 09:39:30+00:00
2,-28.7,152.7,New South Wales,Myobatrachid,2023-01-02 10:30:07+00:00
3,-28.7,152.7,New South Wales,Myobatrachid,2023-01-02 10:30:07+00:00
4,-28.7,152.7,New South Wales,Hylid,2023-01-02 10:30:07+00:00


In [68]:
# Create useful columns from the datetime
# Extract basic time components
df['year'] = df['timestamp_utc'].dt.year
df['month'] = df['timestamp_utc'].dt.month_name()
df['day_of_week'] = df['timestamp_utc'].dt.day_name()
df['hour'] = df['timestamp_utc'].dt.hour
df['day_of_month'] = df['timestamp_utc'].dt.day
df['day_of_year'] = df['timestamp_utc'].dt.dayofyear
df['week_of_year'] = df['timestamp_utc'].dt.isocalendar().week
df['quarter'] = df['timestamp_utc'].dt.quarter
df['is_weekend'] = df['timestamp_utc'].dt.weekday >= 5  # Saturday=5, Sunday=6

# Australian seasons (Southern Hemisphere)
def get_australian_season(month):
    if month in [12, 1, 2]:
        return 'Summer'
    elif month in [3, 4, 5]:
        return 'Autumn'
    elif month in [6, 7, 8]:
        return 'Winter'
    else:  # [9, 10, 11]
        return 'Spring'

df['season'] = df['month'].apply(get_australian_season)

# Time of day categories
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['time_of_day'] = df['hour'].apply(get_time_of_day)

In [72]:
# Remove the UTC column
df = df.drop(df.columns[4], axis=1)
df = df.dropna(subset=['subfamily'])
df.head()

,decimalLatitude,decimalLongitude,stateProvince,subfamily,hour,day_of_month,day_of_year,week_of_year,quarter,is_weekend,season,time_of_day
0,-28.5,153.1,New South Wales,Myobatrachid,0,1,1,52,1,True,Spring,Night
2,-28.7,152.7,New South Wales,Myobatrachid,10,2,2,1,1,False,Spring,Morning
3,-28.7,152.7,New South Wales,Myobatrachid,10,2,2,1,1,False,Spring,Morning
4,-28.7,152.7,New South Wales,Hylid,10,2,2,1,1,False,Spring,Morning
5,-30.4,152.8,New South Wales,Myobatrachid,5,4,4,1,1,False,Spring,Morning


In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 127467 entries, 0 to 136624
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   decimalLatitude   127467 non-null  float64
 1   decimalLongitude  127467 non-null  float64
 2   stateProvince     127467 non-null  object 
 3   subfamily         127467 non-null  object 
 4   hour              127467 non-null  int32  
 5   day_of_month      127467 non-null  int32  
 6   day_of_year       127467 non-null  int32  
 7   week_of_year      127467 non-null  UInt32 
 8   quarter           127467 non-null  int32  
 9   is_weekend        127467 non-null  bool   
 10  season            127467 non-null  object 
 11  time_of_day       127467 non-null  object 
dtypes: UInt32(1), bool(1), float64(2), int32(4), object(4)
memory usage: 9.5+ MB


## Model

In [87]:
df['subfamily'].value_counts()

subfamily
Myobatrachid    80884
Hylid           44807
Toad             1048
Microhylidae      567
Ranid             161
Name: count, dtype: int64

In [95]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Features for the model
feature_columns = ['decimalLatitude', 'decimalLongitude', 'hour', 'day_of_month', 
                   'day_of_year', 'week_of_year', 'quarter', 'is_weekend']

# Add encoded categorical features
le_state = LabelEncoder()
df['stateProvince_encoded'] = le_state.fit_transform(df['stateProvince'])

le_season = LabelEncoder()
df['season_encoded'] = le_season.fit_transform(df['season'])

le_time = LabelEncoder()
df['time_of_day_encoded'] = le_time.fit_transform(df['time_of_day'])

# Final feature set
features = feature_columns + ['stateProvince_encoded', 'season_encoded', 'time_of_day_encoded']

# Prepare data FIRST
df_filtered = df[df['subfamily'].isin([' Myobatrachid', ' Hylid'])]

# THEN add the interaction terms to df_filtered
df_filtered['lat_long_interaction'] = df_filtered['decimalLatitude'] * df_filtered['decimalLongitude']
df_filtered['lat_squared'] = df_filtered['decimalLatitude'] ** 2
features = features + ['lat_long_interaction', 'lat_squared']

X = df_filtered[features]
y = df_filtered['subfamily']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50, stratify=y)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=50)
rf.fit(X_train, y_train)

# Make predictions
y_pred = rf.predict(X_test)

# Evaluate
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("\nFeature Importance:")
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(feature_importance)

/var/folders/7d/cvrlcn892fd5f01pqw2vmql40000gn/T/ipykernel_29904/2830496557.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['lat_long_interaction'] = df_filtered['decimalLatitude'] * df_filtered['decimalLongitude']
/var/folders/7d/cvrlcn892fd5f01pqw2vmql40000gn/T/ipykernel_29904/2830496557.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['lat_squared'] = df_filtered['decimalLatitude'] ** 2


Classification Report:
               precision    recall  f1-score   support

        Hylid       0.60      0.59      0.59      8962
 Myobatrachid       0.77      0.78      0.78     16177

     accuracy                           0.71     25139
    macro avg       0.69      0.69      0.69     25139
 weighted avg       0.71      0.71      0.71     25139


Feature Importance:
                  feature  importance
0         decimalLatitude    0.161680
12            lat_squared    0.154543
1        decimalLongitude    0.143126
11   lat_long_interaction    0.140722
4             day_of_year    0.109966
2                    hour    0.081744
5            week_of_year    0.079388
3            day_of_month    0.052358
10    time_of_day_encoded    0.031793
6                 quarter    0.022700
8   stateProvince_encoded    0.012282
7              is_weekend    0.009697
9          season_encoded    0.000000


In [91]:
rf_optimized = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=50
)

rf_optimized.fit(X_train, y_train)

# Make predictions
y_pred = rf_optimized.predict(X_test)

# Evaluate
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("\nFeature Importance:")
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(feature_importance)

Classification Report:
               precision    recall  f1-score   support

        Hylid       0.60      0.73      0.66      8962
 Myobatrachid       0.83      0.73      0.78     16177

     accuracy                           0.73     25139
    macro avg       0.72      0.73      0.72     25139
 weighted avg       0.75      0.73      0.74     25139


Feature Importance:
                  feature  importance
0         decimalLatitude    0.304974
1        decimalLongitude    0.241594
4             day_of_year    0.117289
2                    hour    0.106545
5            week_of_year    0.078588
3            day_of_month    0.059343
10    time_of_day_encoded    0.033604
8   stateProvince_encoded    0.026357
6                 quarter    0.021748
7              is_weekend    0.009957
9          season_encoded    0.000000


In [93]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.1,
    random_state=50
)

xgb_model.fit(X_train, y_train)

# Make predictions
y_pred = xgb_model.predict(X_test)

# Evaluate
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("\nFeature Importance:")
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(feature_importance)

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/steve/.pyenv/versions/portfolio_project_virtual_env/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <E8D72161-CCD1-3423-9388-36D4CA0A7524> /Users/steve/.pyenv/versions/3.10.13/envs/portfolio_project_virtual_env/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/Syste